In [ ]:
!sudo apt-get install zstd pciutils nano
!git clone https://github.com/VTSTech/LocalClaw
# Install Ollama
!sudo curl -fsSL https://ollama.com/install.sh | sh
# Install cloudflared
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /tmp/cloudflared
!chmod +x /tmp/cloudflared

In [7]:
# 2. Start Ollama in the background
import subprocess, os, time
# We use OLLAMA_HOST=0.0.0.0 so the tunnel can find it
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_API_KEY'] = 'ollama-local'
os.environ['OLLAMA_CONTEXT_LENGTH'] = '262144'
subprocess.Popen(['nohup', 'ollama', 'serve'], stdout=open('ollama.log', 'w'))
time.sleep(5)

In [ ]:
# Start Ollama (if not already running)
#!ollama serve &

import subprocess
import re
import time

# Start cloudflared
proc = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11434'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Wait and capture the tunnel URL
tunnel_url = None
for _ in range(30):  # 30 second timeout
    line = proc.stdout.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[^\s]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(1)

if tunnel_url:
    print(f"✅ Tunnel URL: {tunnel_url}")
else:
    print("❌ Failed to get tunnel URL")

In [ ]:
!rm -rf LocalClaw
!git clone https://github.com/VTSTech/LocalClaw

In [ ]:
!ollama pull qwen2.5-coder:0.5b-instruct-q4_k_m
!ollama pull llama3.2:1b
!ollama pull qwen2-math:1.5b
!ollama pull gemma3:270m
!ollama pull qwen2.5:0.5b
!ollama pull tinyllama:latest
!ollama pull qwen3:0.6b
!ollama pull granite4:350m
!ollama pull granite3.1-moe:1b
!ollama pull smollm:135m
!ollama pull functiongemma:270m
!cat ollama.log